# Linear Regression Deeply

Notebook for **Chapter 6** of the *stats* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/stats/ch06.html

Tip: change a number, predict the output, then run the cell again.

## 6.1 Simple Linear Regression and the OLS Derivation

In [ ]:
import numpy as np
from scipy import stats
x = np.array([1, 2, 3, 4, 5.0]); y = np.array([2, 4, 5, 4, 5.0])
X = np.column_stack([np.ones_like(x), x])
beta = np.linalg.solve(X.T @ X, X.T @ y)           # normal equations
e = y - X @ beta
n, p = X.shape
s2 = e @ e / (n - p)
cov = s2 * np.linalg.inv(X.T @ X)
se = np.sqrt(np.diag(cov)); t = beta / se
print("beta:", beta, " SSE:", round(e @ e, 3), " R2:", round(1 - e @ e / ((y - y.mean()) ** 2).sum(), 3))
print("SE:", np.round(se, 4), " t:", np.round(t, 3), " p:", np.round(2 * stats.t.sf(abs(t), n - p), 4))
print("X'e (should be ~0):", np.round(X.T @ e, 12))
print("lstsq check:", np.linalg.lstsq(X, y, rcond=None)[0])

beta: [2.2 0.6]  SSE: 2.4  R2: 0.6
SE: [0.9381 0.2828]  t: [2.345 2.121]  p: [0.1007 0.124 ]
X'e (should be ~0): [0. 0.]
lstsq check: [2.2 0.6]


In [ ]:
def ols(X, y, names=None):
    X = np.asarray(X, float); y = np.asarray(y, float)
    n, p = X.shape
    XtXi = np.linalg.inv(X.T @ X)
    beta = XtXi @ X.T @ y
    fit = X @ beta; e = y - fit
    sse = e @ e; sst = ((y - y.mean()) ** 2).sum()
    s2 = sse / (n - p)
    se = np.sqrt(s2 * np.diag(XtXi))
    t = beta / se; pv = 2 * stats.t.sf(abs(t), n - p)
    h = np.einsum("ij,jk,ik->i", X, XtXi, X)              # leverage = diag of hat matrix
    r2 = 1 - sse / sst; adj = 1 - (1 - r2) * (n - 1) / (n - p)
    F = ((sst - sse) / (p - 1)) / s2; pF = stats.f.sf(F, p - 1, n - p)
    cooks = e**2 / (p * s2) * h / (1 - h) ** 2
    return dict(beta=beta, se=se, t=t, p=pv, resid=e, fitted=fit, h=h, r2=r2, adj=adj, F=F, pF=pF,
                s=np.sqrt(s2), n=n, k=p, cooks=cooks, XtXi=XtXi, sse=sse, names=names)
def show(m):
    print(f"{'term':12}{'coef':>10}{'se':>9}{'t':>8}{'p':>10}")
    for nm, b, s_, t_, p_ in zip(m["names"], m["beta"], m["se"], m["t"], m["p"]):
        print(f"{nm:12}{b:10.4f}{s_:9.4f}{t_:8.2f}{p_:10.4f}")
    print(f"R2={m['r2']:.4f} adjR2={m['adj']:.4f} F={m['F']:.1f} (p={m['pF']:.2e}) resid SE={m['s']:.2f} n={m['n']}")

rng = np.random.default_rng(60)
n = 200
orders = rng.uniform(100, 1000, n)
dmin = 80 + 0.15 * orders + rng.normal(0, 25, n)
X1 = np.column_stack([np.ones(n), orders])
m1 = ols(X1, dmin, ["intercept", "orders"])
show(m1)
tcrit = stats.t.ppf(0.975, n - 2)
print("95%% CI slope: (%.4f, %.4f)" % (m1["beta"][1] - tcrit * m1["se"][1], m1["beta"][1] + tcrit * m1["se"][1]))
x0 = np.array([1, 800.0]); yhat0 = x0 @ m1["beta"]; q = x0 @ m1["XtXi"] @ x0
print("prediction at 800 orders: %.1f min" % yhat0)
print("CI for the mean   : +/- %.1f" % (tcrit * m1["s"] * np.sqrt(q)))
print("prediction interval: +/- %.1f" % (tcrit * m1["s"] * np.sqrt(1 + q)))

term              coef       se       t         p
intercept      78.1250   3.7600   20.78    0.0000
orders          0.1564   0.0061   25.71    0.0000
R2=0.7695 adjR2=0.7684 F=661.2 (p=5.07e-65) resid SE=23.65 n=200
95% CI slope: (0.1444, 0.1684)
prediction at 800 orders: 203.3 min
CI for the mean   : +/- 4.4
prediction interval: +/- 46.8


In [ ]:
# plot
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6.2, 3.8))
ax.scatter(orders, dmin, s=10, alpha=.6, color="#1f77b4", label="days")
xs = np.linspace(100, 1000, 100)
Xs = np.column_stack([np.ones(100), xs]); ys = Xs @ m1["beta"]
sem = m1["s"] * np.sqrt(np.einsum("ij,jk,ik->i", Xs, m1["XtXi"], Xs))
sep = m1["s"] * np.sqrt(1 + np.einsum("ij,jk,ik->i", Xs, m1["XtXi"], Xs))
ax.plot(xs, ys, color="k", label="OLS line")
ax.fill_between(xs, ys - tcrit * sem, ys + tcrit * sem, color="#d62728", alpha=.5, label="95% CI of mean")
ax.fill_between(xs, ys - tcrit * sep, ys + tcrit * sep, color="gray", alpha=.2, label="95% prediction interval")
ax.set_xlabel("orders waiting"); ax.set_ylabel("delivery time (min)"); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{IMG}/ch06_fit.png", dpi=120); plt.close()

## 6.2 Multiple Regression: Coefficients, t-tests, the F-test, R² and Adjusted R²

In [ ]:
rng = np.random.default_rng(61)
n = 200
orders = rng.uniform(100, 1000, n)
distance = rng.lognormal(3.0, 0.6, n)                   # km
express = rng.binomial(1, 0.6, n)
dmin = 80 + 0.15 * orders + 0.8 * distance - 40 * express + rng.normal(0, 25, n)
X3 = np.column_stack([np.ones(n), orders, distance, express])
m3 = ols(X3, dmin, ["intercept", "orders", "distance_km", "express"])
show(m3)
junk = rng.normal(size=n)
m4 = ols(np.column_stack([X3, junk]), dmin, ["intercept", "orders", "distance_km", "express", "noise_col"])
print("--- adding a pure-noise column ---")
show(m4)
print("R2 %.5f -> %.5f   adjR2 %.5f -> %.5f" % (m3["r2"], m4["r2"], m3["adj"], m4["adj"]))

term              coef       se       t         p
intercept      88.5149   5.1144   17.31    0.0000
orders          0.1381   0.0063   21.74    0.0000
distance_km     0.7505   0.0930    8.07    0.0000
express       -43.0774   3.5524  -12.13    0.0000
R2=0.7872 adjR2=0.7840 F=241.7 (p=1.36e-65) resid SE=24.56 n=200
--- adding a pure-noise column ---
term              coef       se       t         p
intercept      88.9722   5.0956   17.46    0.0000
orders          0.1377   0.0063   21.78    0.0000
distance_km     0.7414   0.0927    8.00    0.0000
express       -43.1175   3.5347  -12.20    0.0000
noise_col       2.8642   1.6584    1.73    0.0857
R2=0.7904 adjR2=0.7861 F=183.9 (p=5.26e-65) resid SE=24.44 n=200
R2 0.78723 -> 0.79044   adjR2 0.78397 -> 0.78614


In [ ]:
m_small = ols(np.column_stack([np.ones(n), orders]), dmin, ["i", "orders"])
q = 2
F = ((m_small["sse"] - m3["sse"]) / q) / (m3["sse"] / (n - 4))
print("nested F(%d,%d) = %.1f  p = %.2e" % (q, n - 4, F, stats.f.sf(F, q, n - 4)))
m_cache = ols(np.column_stack([np.ones(n), orders, distance]), dmin, ["i", "u", "p"])
F1 = ((m_cache["sse"] - m3["sse"]) / 1) / (m3["sse"] / (n - 4))
print("one-variable check: F=%.2f  t^2=%.2f" % (F1, m3["t"][3] ** 2))
ml = ols(np.column_stack([np.ones(n), np.log(orders)]), np.log(dmin), ["i", "log_users"])
print("elasticity of delivery time w.r.t. orders: %.3f  (se %.3f)" % (ml["beta"][1], ml["se"][1]))

nested F(2,196) = 108.8  p = 1.65e-32
one-variable check: F=147.05  t^2=147.05
elasticity of delivery time w.r.t. orders: 0.451  (se 0.034)


## 6.3 The Assumptions, and How to Check Them

In [ ]:
rng = np.random.default_rng(62)
n = 300
u = rng.uniform(100, 1000, n)
y_h = 80 + 0.15 * u + rng.normal(0, 1, n) * 0.05 * u
Xh = np.column_stack([np.ones(n), u]); mh = ols(Xh, y_h, ["intercept", "orders"])
e2 = mh["resid"] ** 2
aux = ols(Xh, e2, ["i", "u"])
LM = n * aux["r2"]
print("Breusch-Pagan: LM = %.1f  p = %.2e" % (LM, stats.chi2.sf(LM, 1)))
def hc3(m, X):
    X = np.asarray(X, float)
    meat = X.T @ (X * (m["resid"] ** 2 / (1 - m["h"]) ** 2)[:, None])
    return np.sqrt(np.diag(m["XtXi"] @ meat @ m["XtXi"]))
print("slope %.4f   naive SE %.4f   HC3 SE %.4f" % (mh["beta"][1], mh["se"][1], hc3(mh, Xh)[1]))
ml = ols(np.column_stack([np.ones(n), np.log(u)]), np.log(np.maximum(y_h, 1)), ["i", "log_u"])
aux2 = ols(np.column_stack([np.ones(n), np.log(u)]), ml["resid"] ** 2, ["i", "x"])
print("after log-log: BP LM = %.1f  p = %.1e" % (n * aux2["r2"], stats.chi2.sf(n * aux2["r2"], 1)))

Breusch-Pagan: LM = 47.3  p = 6.23e-12
slope 0.1491   naive SE 0.0070   HC3 SE 0.0066
after log-log: BP LM = 13.0  p = 3.2e-04


In [ ]:
rng = np.random.default_rng(63)
n = 200
u = np.sort(rng.uniform(100, 1000, n))
y_c = 100 + 0.0004 * u**2 + rng.normal(0, 15, n)
mline = ols(np.column_stack([np.ones(n), u]), y_c, ["i", "u"])
mquad = ols(np.column_stack([np.ones(n), u, u**2]), y_c, ["i", "u", "u2"])
uc2 = (u - u.mean()) ** 2     # centred square: the 'bend' term
print("line R2 %.3f; corr(resid_line, centred u^2) = %.3f" % (mline["r2"], np.corrcoef(mline["resid"], uc2)[0, 1]))
print("quad R2 %.3f; corr(resid_quad, centred u^2) = %.3f; coef on u^2 = %.6f" % (mquad["r2"], np.corrcoef(mquad["resid"], uc2)[0, 1], mquad["beta"][2]))
print("Shapiro-Wilk on quadratic-model residuals: p = %.3f" % stats.shapiro(mquad["resid"]).pvalue)
# AR(1) noise
innov = rng.normal(0, 10, n); noise = np.zeros(n)
for t in range(1, n): noise[t] = 0.8 * noise[t - 1] + innov[t]
days = np.arange(n, dtype=float)
y_a = 50 + 0.2 * days + noise
ma = ols(np.column_stack([np.ones(n), days]), y_a, ["i", "day"])
dw = np.sum(np.diff(ma["resid"]) ** 2) / np.sum(ma["resid"] ** 2)
iid = ols(np.column_stack([np.ones(n), days]), 50 + 0.2 * days + rng.normal(0, 17, n), ["i", "day"])
print("Durbin-Watson, AR(1) noise: %.2f   independent noise: %.2f" % (dw, np.sum(np.diff(iid["resid"]) ** 2) / np.sum(iid["resid"] ** 2)))

line R2 0.938; corr(resid_line, centred u^2) = 0.832
quad R2 0.982; corr(resid_quad, centred u^2) = -0.000; coef on u^2 = 0.000396
Shapiro-Wilk on quadratic-model residuals: p = 0.585
Durbin-Watson, AR(1) noise: 0.51   independent noise: 1.79


In [ ]:
# plot
fig, ax = plt.subplots(1, 3, figsize=(10, 3.2))
ax[0].scatter(mquad["fitted"], mquad["resid"], s=8, color="#2ca02c"); ax[0].axhline(0, color="k"); ax[0].set_title("Good: random band")
ax[1].scatter(mh["fitted"], mh["resid"], s=8, color="#d62728"); ax[1].axhline(0, color="k"); ax[1].set_title("Funnel: unequal variance")
ax[2].scatter(mline["fitted"], mline["resid"], s=8, color="#1f77b4"); ax[2].axhline(0, color="k"); ax[2].set_title("Curve: missed non-linearity")
for a_ in ax: a_.set_xlabel("fitted value"); a_.set_ylabel("residual")
plt.tight_layout(); plt.savefig(f"{IMG}/ch06_residuals.png", dpi=120); plt.close()

## 6.4 Multicollinearity and the Variance Inflation Factor (VIF)

In [ ]:
rng = np.random.default_rng(64)
n = 200
orders = rng.uniform(100, 1000, n)
distance = rng.lognormal(3.0, 0.6, n); express = rng.binomial(1, 0.6, n)
dmin = 80 + 0.15 * orders + 0.8 * distance - 40 * express + rng.normal(0, 25, n)
vans = 0.05 * orders + rng.normal(0, 1, n)
Xa = np.column_stack([np.ones(n), orders, distance, express])
Xb = np.column_stack([np.ones(n), orders, distance, express, vans])
ma_ = ols(Xa, dmin, ["intercept", "orders", "distance", "express"]); mb_ = ols(Xb, dmin, ["intercept", "orders", "distance", "express", "vans"])
show(mb_)
def vif(X, j):
    others = np.delete(X, j, axis=1)
    r2 = ols(others, X[:, j])["r2"] if True else 0
    return 1 / (1 - r2)
for j, nm in [(1, "orders"), (2, "distance"), (3, "express"), (4, "vans")]:
    print("VIF %-8s = %8.1f" % (nm, vif(Xb, j)))
print("SE(orders) without vans: %.5f   with vans: %.5f   ratio %.1f   sqrt(VIF) %.1f" % (ma_["se"][1], mb_["se"][1], mb_["se"][1] / ma_["se"][1], np.sqrt(vif(Xb, 1))))
print("R2 without vans %.4f, with vans %.4f" % (ma_["r2"], mb_["r2"]))

term              coef       se       t         p
intercept      70.8093   6.0435   11.72    0.0000
orders          0.2647   0.0855    3.10    0.0022
distance        0.9445   0.1182    7.99    0.0000
express       -37.7678   3.6267  -10.41    0.0000
vans           -2.2543   1.7072   -1.32    0.1882
R2=0.7611 adjR2=0.7562 F=155.3 (p=1.77e-59) resid SE=24.22 n=200
VIF orders   =    158.9
VIF distance =      1.0
VIF express  =      1.0
VIF vans     =    158.7
SE(orders) without vans: 0.00685   with vans: 0.08546   ratio 12.5   sqrt(VIF) 12.6
R2 without vans 0.7590, with vans 0.7611


In [ ]:
est_a, est_b, pred_a, pred_b = [], [], [], []
for _ in range(200):
    u = rng.uniform(100, 1000, n); pl = rng.lognormal(3.0, 0.6, n); ca = rng.binomial(1, 0.6, n)
    y_ = 80 + 0.15 * u + 0.8 * pl - 40 * ca + rng.normal(0, 25, n); cp = 0.05 * u + rng.normal(0, 1, n)
    A = np.column_stack([np.ones(n), u, pl, ca]); B = np.column_stack([A, cp])
    ba = np.linalg.lstsq(A, y_, rcond=None)[0]; bb = np.linalg.lstsq(B, y_, rcond=None)[0]
    est_a.append(ba[1]); est_b.append(bb[1])
    pred_a.append(np.array([1, 500, 20, 1]) @ ba); pred_b.append(np.array([1, 500, 20, 1, 25]) @ bb)
print("orders coef without vans: mean %.4f sd %.4f" % (np.mean(est_a), np.std(est_a)))
print("orders coef with vans   : mean %.4f sd %.4f  share negative: %.2f" % (np.mean(est_b), np.std(est_b), np.mean(np.array(est_b) < 0)))
print("prediction at a typical point: sd without %.2f, with %.2f" % (np.std(pred_a), np.std(pred_b)))

orders coef without vans: mean 0.1495 sd 0.0069
orders coef with vans   : mean 0.1601 sd 0.0936  share negative: 0.06
prediction at a typical point: sd without 2.34, with 2.34


## 6.5 Leverage, Influential Points and Cook's Distance

In [ ]:
rng = np.random.default_rng(65)
n = 200
u = rng.uniform(100, 1000, n); y_ = 80 + 0.15 * u + rng.normal(0, 25, n)
u2 = np.append(u, 5000.0); y2 = np.append(y_, 300.0)
mA = ols(np.column_stack([np.ones(n), u]), y_, ["i", "u"])
mB = ols(np.column_stack([np.ones(n + 1), u2]), y2, ["i", "u"])
print("slope without the holiday-sale day: %.4f   with: %.4f" % (mA["beta"][1], mB["beta"][1]))
i = n
print("holiday-sale day: leverage %.3f   Cook's D %.2f   residual %.1f" % (mB["h"][i], mB["cooks"][i], mB["resid"][i]))
print("mean leverage %.5f  (p/n = %.5f)   points with h > 2p/n: %d" % (mB["h"].mean(), 2 / (n + 1), (mB["h"] > 4 / (n + 1)).sum()))
top = np.argsort(mB["cooks"])[::-1][:3]
print("top 3 Cook's D:", [(int(j), round(float(mB["cooks"][j]), 3)) for j in top], "  4/n =", round(4 / (n + 1), 4))

slope without the holiday-sale day: 0.1564   with: 0.0813
holiday-sale day: leverage 0.602   Cook's D 74.21   residual -221.7
mean leverage 0.00995  (p/n = 0.00995)   points with h > 2p/n: 1
top 3 Cook's D: [(200, 74.206), (34, 0.031), (183, 0.029)]   4/n = 0.0199


In [ ]:
u3 = np.append(u, 550.0); y3 = np.append(y_, 80 + 0.15 * 550 + 150)
mC = ols(np.column_stack([np.ones(n + 1), u3]), y3, ["i", "u"])
print("slope without: %.4f  with the middle outlier: %.4f" % (mA["beta"][1], mC["beta"][1]))
print("middle outlier: leverage %.4f  Cook's D %.3f  residual %.1f  (s = %.1f)" % (mC["h"][n], mC["cooks"][n], mC["resid"][n], mC["s"]))

slope without: 0.1564  with the middle outlier: 0.1565
middle outlier: leverage 0.0050  Cook's D 0.075  residual 150.3  (s = 27.5)


## 6.6 Interpreting Regression Honestly: Causation, Extrapolation and Omitted Variables

In [ ]:
rng = np.random.default_rng(66)
n = 100_000
x1 = rng.standard_normal(n); x2 = 0.7 * x1 + np.sqrt(1 - 0.49) * rng.standard_normal(n)
y = 1.0 * x1 + 2.0 * x2 + rng.standard_normal(n)
full = np.linalg.lstsq(np.column_stack([np.ones(n), x1, x2]), y, rcond=None)[0]
short = np.linalg.lstsq(np.column_stack([np.ones(n), x1]), y, rcond=None)[0]
print("full model :", np.round(full, 3))
print("short model:", np.round(short, 3), " (theory for slope: 1 + 2*0.7 = 2.4)")

full model : [1.000e-03 9.970e-01 2.002e+00]
short model: [-0.007  2.394]  (theory for slope: 1 + 2*0.7 = 2.4)


In [ ]:
rng = np.random.default_rng(67)
n = 200
u = rng.uniform(100, 1000, n); y_ = 80 + 0.15 * u + rng.normal(0, 25, n)
m = ols(np.column_stack([np.ones(n), u]), y_, ["i", "u"])
pred = m["beta"] @ np.array([1, 3000.0])
truth = 80 + 0.15 * 1000 + 0.9 * (3000 - 1000)
print("linear model prediction at 3000 orders: %.0f min   truth: %.0f min" % (pred, truth))

linear model prediction at 3000 orders: 508 min   truth: 2030 min


In [ ]:
# not run here
import statsmodels.api as sm
X = sm.add_constant(df[["orders", "distance_km", "express"]])
res = sm.OLS(df["delivery time"], X).fit()                  # classical SE
print(res.summary())                                   # coef, SE, t, p, R2, adj R2, F, DW, Omnibus
res_robust = sm.OLS(df["delivery time"], X).fit(cov_type="HC3")
infl = res.get_influence(); print(infl.cooks_distance[0].max())
from statsmodels.stats.outliers_influence import variance_inflation_factor
print([variance_inflation_factor(X.values, i) for i in range(X.shape[1])])

## Chapter recap

## Practice set

Problem 6.7.1 Points (1, 3), (2, 5), (3, 7). What is the OLS line?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/stats/ch06.html